# Integers in binary

The answers to Section 11 of the sample questions, `documentation/tex/questions`, run.
The questions concern Appendix A of the lecture notes, an integer as a set of bit positions,
and the two books of Section 1.2.4 that hold the occupied prices of a side as one integer,
`TickArrayBook` and `BitmapBook`.

Each question's code is copied here verbatim, wrapped in a function where it prints.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
Prices are integer counts of ticks.

In [ ]:
import contextlib
import io
import timeit

import numpy as np
import pandas as pd
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.hawkes import ExponentialHawkes, HawkesParams
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook

from unito26.lob.binary_gaps import count_binary_gaps, measure_largest_binary_gap
from unito26.lob.messages import ReportedDepth
from unito26.lob.orderbook import BitmapBook, TickArrayBook


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")

## `question.tickArrayIntegersAfterACrossingBuy`: the two integers after a crossing buy

The integers before the buy are printed too: 13 on the bids, 26 on the asks.

In [ ]:
def run():
    book = TickArrayBook(floor=200, width=10)
    for price, size in ((203, 50), (202, 40), (200, 70)):
        book.set_size(BUY, price, size)
    for price, size in ((205, 60), (206, 30), (208, 90)):
        book.set_size(SELL, price, size)
    book.apply(limit_order(1.0, 100, 206, BUY), record=False)
    print(book._bits[BUY], book._bits[SELL],
          book.best_bid_price, book.best_ask_price)


before = TickArrayBook(floor=200, width=10)
for price, size in ((203, 50), (202, 40), (200, 70)):
    before.set_size(BUY, price, size)
for price, size in ((205, 60), (206, 30), (208, 90)):
    before.set_size(SELL, price, size)
print("before:", before._bits[BUY], before._bits[SELL])
assert (before._bits[BUY], before._bits[SELL]) == (13, 26)

printed = outcome(run)
print("after: ", printed)
verify("question.tickArrayIntegersAfterACrossingBuy", "C", [
    printed == "77 256 206 208",
    printed == "77 4 206 208",
    printed == "77 2 206 208",
    printed == "13 2 203 208",
    printed == "77 26 206 205",
])

## `question.bitmapBestAskFromTheWrongEnd`: the highest ask, and an empty side

The package's own `BitmapBook` is run as well: it prints `101 None None`.

In [ ]:
class Book(BitmapBook):
    def best_price(self, direction):
        return self.floor + self._bits[direction].bit_length() - 1

BookA = Book

In [ ]:
class Book(BitmapBook):
    def best_price(self, direction):
        bits = self._bits[direction]
        if direction == BUY:
            return self.floor + bits.bit_length() - 1
        return self.floor + (bits & -bits).bit_length() - 1

BookB = Book

In [ ]:
def run(Book):
    book = Book(floor=95)
    for price, size in ((98, 40), (97, 60)):
        book.set_size(BUY, price, size)
    for price, size in ((101, 30), (102, 50), (105, 20)):
        book.set_size(SELL, price, size)
    before = book.best_ask_price
    book.apply(market_order(1.0, 100, BUY), record=False)
    print(before, book.best_ask_price, book.spread)


a, b = outcome(run, BookA), outcome(run, BookB)
print("A:", a, "   B:", b, "   package:", outcome(run, BitmapBook))
verify("question.bitmapBestAskFromTheWrongEnd", "B", [
    a == b == "101 None None",
    a == "105 94 -4" and b == "101 94 -4",
    a == "105 None None" and b == "101 None None",
    a == b == "101 94 -4",
    a == "105 95 -3" and b == "101 95 -3",
])

## `question.occupancyWrittenByAddition`: an addition carries where an or does not

The package's own `TickArrayBook` is run as well: it prints `99 60`.

In [ ]:
class Book(TickArrayBook):
    def set_size(self, direction, price, size):
        self._sizes[direction][price - self.floor] = size
        bit = 1 << (price - self.floor if direction == BUY
                    else self.ceiling - price)
        if size > 0:
            self._bits[direction] += bit
        else:
            self._bits[direction] -= bit

BookA = Book

In [ ]:
class Book(TickArrayBook):
    def set_size(self, direction, price, size):
        sizes = self._sizes[direction]
        slot = price - self.floor
        bit = 1 << (slot if direction == BUY else self.ceiling - price)
        if (sizes[slot] > 0) != (size > 0):
            self._bits[direction] ^= bit
        sizes[slot] = size

BookB = Book

In [ ]:
def run(Book):
    book = Book(floor=95, width=12)
    for price, size in ((99, 100), (98, 60)):
        book.set_size(BUY, price, size)
    book.apply(market_order(1.0, 40, SELL), record=False)
    print(book.best_bid_price, book.best_bid_size)


a, b = outcome(run, BookA), outcome(run, BookB)
print("A:", a, "   B:", b, "   package:", outcome(run, TickArrayBook))
verify("question.occupancyWrittenByAddition", "D", [
    a == b == "99 60",
    a == "99 60" and b == "98 60",
    a == "100 60" and b == "99 60",
    a == "100 0" and b == "99 60",
    a == "100 0" and b == "98 60",
])

## `question.occupancyClearedWithMinusForTilde`: `&= -bit` keeps the bit and clears those below

The package's own `TickArrayBook` is run on the same messages: it prints
`99 [(99, 100), (96, 40)]`.

In [ ]:
def run():
    class Book(TickArrayBook):
        def set_size(self, direction, price, size):
            self._sizes[direction][price - self.floor] = size
            bit = 1 << (price - self.floor if direction == BUY
                        else self.ceiling - price)
            if size > 0:
                self._bits[direction] |= bit
            else:
                self._bits[direction] &= -bit

    book = Book(floor=95, width=12)
    for price, size in ((99, 100), (98, 60), (96, 40)):
        book.set_size(BUY, price, size)
    book.apply(withdrawal(1.0, 60, 98, BUY), record=False)
    print(book.best_bid_price, book.occupied_levels(BUY, ReportedDepth(3)))


def package():
    book = TickArrayBook(floor=95, width=12)
    for price, size in ((99, 100), (98, 60), (96, 40)):
        book.set_size(BUY, price, size)
    book.apply(withdrawal(1.0, 60, 98, BUY), record=False)
    print(book.best_bid_price, book.occupied_levels(BUY, ReportedDepth(3)))


assert 26 & -8 == 24 and 26 & ~8 == 18 and ~8 == -9
printed = outcome(run)
print(printed, "   package:", outcome(package))
verify("question.occupancyClearedWithMinusForTilde", "E", [
    printed == "98 [(98, 0)]",
    printed == "99 [(99, 100), (98, 0), (96, 40)]",
    printed == "99 [(99, 100), (96, 40)]",
    printed == "99 [(99, 100)]",
    printed == "99 [(99, 100), (98, 0)]",
])

## `question.gapsOfTheUncutBidOccupancy`: trailing zeros are not a gap

The package's own statistics, read on the window `span_bits` cuts, are printed beside:
one gap, of two ticks.

In [ ]:
def run():
    book = TickArrayBook(floor=95, width=12)
    for price, size in ((104, 10), (103, 20), (100, 30), (99, 40)):
        book.set_size(BUY, price, size)
    bits = book._bits[BUY]
    print(count_binary_gaps(bits), bin(bits)[2:].count("10"),
          measure_largest_binary_gap(bits))


book = TickArrayBook(floor=95, width=12)
for price, size in ((104, 10), (103, 20), (100, 30), (99, 40)):
    book.set_size(BUY, price, size)
print("integer:", bin(book._bits[BUY]), "   window:", bin(book.span_bits(BUY, ReportedDepth(4))),
      "   package:", book.gap_count(BUY, ReportedDepth(4)),
      book.largest_gap_size_between_non_empty_levels(BUY, ReportedDepth(4)))

printed = outcome(run)
print(printed)
verify("question.gapsOfTheUncutBidOccupancy", "A", [
    printed == "1 2 4",
    printed == "1 1 2",
    printed == "1 2 2",
    printed == "1 1 4",
    printed == "2 2 4",
])

## `question.askWindowsOfTheTwoBitmaps`: the window from the low end and from the high end

The two ask integers are printed first: the asks counted up from the floor in `BitmapBook`,
down from the ceiling in `TickArrayBook`.

In [ ]:
def run():
    asks = {101: 30, 102: 50, 105: 20, 106: 40}
    bitmap = BitmapBook(floor=95)
    array = TickArrayBook(floor=95, width=12)
    for price, size in asks.items():
        bitmap.set_size(SELL, price, size)
        array.set_size(SELL, price, size)
    depth = ReportedDepth(3)
    print(bin(bitmap.span_bits(SELL, depth)),
          bin(array.span_bits(SELL, depth)))


bitmap = BitmapBook(floor=95)
array = TickArrayBook(floor=95, width=12)
for price, size in {101: 30, 102: 50, 105: 20, 106: 40}.items():
    bitmap.set_size(SELL, price, size)
    array.set_size(SELL, price, size)
print("ask integers:", bin(bitmap._bits[SELL]), bin(array._bits[SELL]))

printed = outcome(run)
print("windows:     ", printed)
verify("question.askWindowsOfTheTwoBitmaps", "D", [
    printed == "0b11001 0b11001",
    printed == "0b10011 0b10011",
    printed == "0b110011 0b110011",
    printed == "0b10011 0b11001",
    printed == "0b11001 0b10011",
])

## `question.lowestSetBitCostsTheBand`: the highest set bit against the lowest

Each best ask is read 2000 times, five times over, at each width, and the best of the five
is kept.  The choices are read off the growth of each time from $W = 10^3$ to $W = 10^6$,
the growth of the bitmap's from $10^5$ to $10^6$, and the ratio of the two at $10^6$.

In [ ]:
def timed(width):
    bitmap = BitmapBook(floor=0)
    array = TickArrayBook(floor=0, width=width)
    for price in (1, width - 1):
        bitmap.set_size(SELL, price, 100)
        array.set_size(SELL, price, 100)
    assert bitmap.best_ask_price == array.best_ask_price == 1
    repeats = 2_000
    per_read = {}
    for name, book in (("array", array), ("bitmap", bitmap)):
        best = min(timeit.repeat(lambda: book.best_ask_price, number=repeats, repeat=5))
        per_read[name] = best / repeats
    return per_read


timings = {}
for width in (10**3, 10**4, 10**5, 10**6):
    timings[width] = timed(width)
    print(f"W = {width:>8}:  array {timings[width]['array'] * 1e9:8.1f} ns   "
          f"bitmap {timings[width]['bitmap'] * 1e9:10.1f} ns per read")

growth_array = timings[10**6]["array"] / timings[10**3]["array"]
growth_bitmap = timings[10**6]["bitmap"] / timings[10**3]["bitmap"]
growth_bitmap_last = timings[10**6]["bitmap"] / timings[10**5]["bitmap"]
bitmap_over_array = timings[10**6]["bitmap"] / timings[10**6]["array"]
print(f"growth from 10^3 to 10^6: array {growth_array:.1f}x, bitmap {growth_bitmap:.0f}x; "
      f"bitmap from 10^5 to 10^6 {growth_bitmap_last:.1f}x; "
      f"bitmap over array at 10^6 {bitmap_over_array:.0f}x")
verify("question.lowestSetBitCostsTheBand", "A", [
    growth_array < 2 and growth_bitmap_last > 5 and bitmap_over_array > 5,
    growth_array < 2 and growth_bitmap < 2,
    growth_array > 5 and growth_bitmap > 5,
    growth_bitmap < 2 and growth_array > 5,
])